In [1]:
# ============================================================
# PROJET DE FIN DE MODULE — EMSI 2025-2026
# Module : Prétraitement, représentation, modélisation et analyse
#          des données avec scikit-learn
# ============================================================
# Ce fichier contient le code exécutable du projet.
# L'analyse détaillée se trouve dans le rapport académique (PDF).
# ============================================================

# ============================================================
# 1. IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import seaborn as sns
from datetime import datetime
import json
import os
import zipfile
from collections import Counter
warnings.filterwarnings('ignore')

# Scikit-learn imports
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, MaxAbsScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer, HashingVectorizer
from sklearn.datasets import fetch_20newsgroups

# Configuration
plt.rcParams['figure.figsize'] = (10, 5)
RANDOM_STATE = 42

print("✅ Tous les imports sont OK\n")


# ============================================================
# 2. PARTIE I : TITANIC (Données tabulaires)
# ============================================================
print("="*60)
print("PARTIE I : ANALYSE DU DATASET TITANIC")
print("="*60)

# ------------------------------------------------------------------
# 2.1 Chargement et inspection des données
# ------------------------------------------------------------------
df_raw = sns.load_dataset('titanic')
COLS_KEEP = ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'alone']
df = df_raw[COLS_KEEP].copy()
df['alone'] = df['alone'].astype(int)

print(f"\n📊 Dimensions : {df.shape[0]} lignes × {df.shape[1]} colonnes")
print(f"🎯 Variable cible : 'survived' (0 = décédé, 1 = survivant)")

# Affichage des types et valeurs manquantes
print("\n📋 Types de données :")
print(df.dtypes)

print("\n🔍 Valeurs manquantes :")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
for col in missing[missing > 0].index:
    print(f"   - {col}: {missing[col]} ({missing_pct[col]}%)")

# ------------------------------------------------------------------
# 2.2 Préparation des données (train/test split)
# ------------------------------------------------------------------
FEATURES = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'alone']
TARGET = 'survived'

X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"\n📁 Split des données :")
print(f"   - Train : {X_train.shape[0]} échantillons")
print(f"   - Test  : {X_test.shape[0]} échantillons")

# ------------------------------------------------------------------
# 2.3 Définition des groupes de colonnes
# ------------------------------------------------------------------
NUM_COLS = ['age', 'fare', 'sibsp', 'parch']      # Variables numériques
ORD_COLS = ['pclass']                              # Variable ordinale
NOM_COLS = ['sex', 'embarked']                     # Variables nominales
BIN_COLS = ['alone']                               # Variable binaire

print("\n🏷️ Classification des variables :")
print(f"   - Numériques : {NUM_COLS}")
print(f"   - Ordinale   : {ORD_COLS}")
print(f"   - Nominale   : {NOM_COLS}")
print(f"   - Binaire    : {BIN_COLS}")

# ------------------------------------------------------------------
# 2.4 Construction du préprocesseur (ColumnTransformer)
# ------------------------------------------------------------------
# Branche numérique : imputation médiane + standardisation
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Branche ordinale : imputation mode + encodage ordinal
ord_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[[1, 2, 3]]))
])

# Branche nominale : imputation mode + one-hot encoding
nom_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, NUM_COLS),
    ('ord', ord_pipeline, ORD_COLS),
    ('nom', nom_pipeline, NOM_COLS),
    ('bin', 'passthrough', BIN_COLS)
])

print("\n⚙️ Préprocesseur construit avec ColumnTransformer")

# ------------------------------------------------------------------
# 2.5 Définition des pipelines
# ------------------------------------------------------------------
# Pipeline A : Logistic Regression (baseline)
pipe_logreg = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', LogisticRegression(max_iter=500, random_state=RANDOM_STATE))
])

# Pipeline B : Random Forest
pipe_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))
])

# Pipeline C : Random Forest avec KNNImputer
num_pipeline_knn = Pipeline([
    ('imputer', KNNImputer(n_neighbors=5)),
    ('scaler', StandardScaler())
])
preprocessor_knn = ColumnTransformer([
    ('num', num_pipeline_knn, NUM_COLS),
    ('ord', ord_pipeline, ORD_COLS),
    ('nom', nom_pipeline, NOM_COLS),
    ('bin', 'passthrough', BIN_COLS)
])
pipe_knn = Pipeline([
    ('preprocessor', preprocessor_knn),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))
])

print("\n🚀 3 pipelines définis :")
print("   - Pipeline A : Logistic Regression (baseline)")
print("   - Pipeline B : Random Forest")
print("   - Pipeline C : Random Forest + KNNImputer")

# ------------------------------------------------------------------
# 2.6 Validation croisée
# ------------------------------------------------------------------
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("\n📊 Validation croisée 5-fold sur X_train :")
results_cv = {}
for name, pipe in [('LogReg', pipe_logreg), ('RandomForest', pipe_rf), ('RF+KNNImputer', pipe_knn)]:
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='accuracy')
    results_cv[name] = scores
    print(f"   {name:15s} : {scores.mean():.4f} ± {scores.std():.4f}")

# ------------------------------------------------------------------
# 2.7 Optimisation des hyperparamètres (GridSearchCV)
# ------------------------------------------------------------------
print("\n🔧 Recherche des meilleurs hyperparamètres pour Random Forest...")
param_grid = {
    'clf__n_estimators': [50, 100, 200],
    'clf__max_depth': [None, 5, 10],
    'clf__min_samples_split': [2, 5]
}

grid_search = GridSearchCV(pipe_rf, param_grid, cv=cv, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

print(f"   ✅ Meilleurs paramètres : {grid_search.best_params_}")
print(f"   ✅ Meilleur score CV : {grid_search.best_score_:.4f}")

best_pipe = grid_search.best_estimator_

# ------------------------------------------------------------------
# 2.8 Évaluation finale sur le test set
# ------------------------------------------------------------------
print("\n📈 Évaluation finale sur X_test :")
for name, pipe in [('LogReg', pipe_logreg), ('RF optimisé', best_pipe), ('RF+KNN', pipe_knn)]:
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
    print(f"   {name:15s} : Accuracy = {acc:.4f}, AUC = {auc:.4f}")

y_pred_best = best_pipe.predict(X_test)


# ============================================================
# 3. PARTIE II : 20 NEWSGROUPS (Données textuelles)
# ============================================================
print("\n" + "="*60)
print("PARTIE II : ANALYSE DU CORPUS 20 NEWSGROUPS")
print("="*60)

# ------------------------------------------------------------------
# 3.1 Chargement du corpus
# ------------------------------------------------------------------
CATEGORIES = ['sci.med', 'sci.space', 'rec.sport.baseball', 'talk.politics.guns']

news_train = fetch_20newsgroups(
    subset='train', categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'),
    random_state=RANDOM_STATE
)

news_test = fetch_20newsgroups(
    subset='test', categories=CATEGORIES,
    remove=('headers', 'footers', 'quotes'),
    random_state=RANDOM_STATE
)

X_text_train, y_text_train = news_train.data, news_train.target
X_text_test, y_text_test = news_test.data, news_test.target

print(f"\n📁 Dimensions du corpus :")
print(f"   - Train : {len(X_text_train)} documents")
print(f"   - Test  : {len(X_text_test)} documents")
print(f"   - Classes : {news_train.target_names}")

# ------------------------------------------------------------------
# 3.2 Comparaison des vectoriseurs
# ------------------------------------------------------------------
print("\n🔍 Comparaison des représentations vectorielles :")
vectorizers = {
    'CountVectorizer': CountVectorizer(max_features=10000, stop_words='english'),
    'TF-IDF': TfidfVectorizer(max_features=10000, stop_words='english'),
    'TF-IDF+bigrammes': TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1,2))
}

for name, vec in vectorizers.items():
    X_vec = vec.fit_transform(X_text_train)
    density = X_vec.nnz / (X_vec.shape[0] * X_vec.shape[1]) * 100
    print(f"   {name:20s} : shape={X_vec.shape}, densité={density:.3f}%")

# ------------------------------------------------------------------
# 3.3 Pipelines textuels
# ------------------------------------------------------------------
print("\n🚀 Construction des pipelines textuels :")

pipe_count_nb = Pipeline([
    ('vect', CountVectorizer(max_features=10000, stop_words='english')),
    ('clf', ComplementNB())
])

pipe_tfidf_lr = Pipeline([
    ('vect', TfidfVectorizer(max_features=10000, stop_words='english')),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

pipe_tfidf_bigram_lr = Pipeline([
    ('vect', TfidfVectorizer(max_features=20000, stop_words='english', ngram_range=(1,2))),
    ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

text_pipelines = {
    'Count+NB': pipe_count_nb,
    'TF-IDF+LR': pipe_tfidf_lr,
    'TF-IDF+bigrammes+LR': pipe_tfidf_bigram_lr
}

for name in text_pipelines:
    print(f"   - {name}")

# ------------------------------------------------------------------
# 3.4 Validation croisée
# ------------------------------------------------------------------
print("\n📊 Validation croisée 3-fold :")
cv_text = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

for name, pipe in text_pipelines.items():
    scores = cross_val_score(pipe, X_text_train, y_text_train, cv=cv_text, scoring='accuracy')
    print(f"   {name:25s} : {scores.mean():.4f} ± {scores.std():.4f}")

# ------------------------------------------------------------------
# 3.5 Évaluation finale
# ------------------------------------------------------------------
print("\n📈 Évaluation finale sur le test set :")
best_acc = 0
best_text_pipe = None
best_text_name = None

for name, pipe in text_pipelines.items():
    pipe.fit(X_text_train, y_text_train)
    y_pred = pipe.predict(X_text_test)
    acc = accuracy_score(y_text_test, y_pred)
    print(f"   {name:25s} : Accuracy = {acc:.4f}")
    if acc > best_acc:
        best_acc = acc
        best_text_pipe = pipe
        best_text_name = name

print(f"\n🏆 Meilleur pipeline : {best_text_name} (accuracy={best_acc:.4f})")
y_pred_best_text = best_text_pipe.predict(X_text_test)

# ------------------------------------------------------------------
# 3.6 Analyse des mots discriminants (pour TF-IDF + LR)
# ------------------------------------------------------------------
print("\n🔍 Analyse des mots les plus discriminants :")
pipe_tfidf_lr.fit(X_text_train, y_text_train)
vectorizer = pipe_tfidf_lr.named_steps['vect']
clf = pipe_tfidf_lr.named_steps['clf']
feature_names_text = vectorizer.get_feature_names_out()

for i, class_name in enumerate(news_train.target_names):
    coef = clf.coef_[i]
    top_idx = np.argsort(coef)[-5:]  # Top 5 mots
    top_words = [feature_names_text[j] for j in top_idx]
    print(f"   {class_name.split('.')[-1]:20s} : {top_words}")


# ============================================================
# 4. VISUALISATIONS
# ============================================================
print("\n" + "="*60)
print("GÉNÉRATION DES FIGURES")
print("="*60)

os.makedirs('figures', exist_ok=True)

# Figure 1 : Matrice de confusion - Titanic
fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_best,
                                         display_labels=['Décédé', 'Survivant'],
                                         ax=ax, cmap='Blues')
ax.set_title('Matrice de confusion - Random Forest optimisé')
plt.tight_layout()
plt.savefig('figures/matrice_confusion_titanic.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Figure 1 générée : matrice_confusion_titanic.png")

# Figure 2 : Courbes ROC
fig, ax = plt.subplots(figsize=(8,6))
for name, pipe in [('LogReg', pipe_logreg), ('RF optimisé', best_pipe)]:
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, ax=ax, name=name)
ax.plot([0,1], [0,1], 'k--', label='Aléatoire')
ax.legend(loc='lower right')
ax.set_title('Courbes ROC comparées - Titanic')
plt.tight_layout()
plt.savefig('figures/courbes_ROC.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Figure 2 générée : courbes_ROC.png")

# Figure 3 : Importance des variables
perm_imp = permutation_importance(best_pipe, X_test, y_test, n_repeats=20, random_state=RANDOM_STATE)
perm_df = pd.DataFrame({
    'feature': FEATURES,
    'importance_mean': perm_imp.importances_mean,
    'importance_std': perm_imp.importances_std
}).sort_values('importance_mean', ascending=True)

fig, ax = plt.subplots(figsize=(8,5))
ax.barh(perm_df['feature'], perm_df['importance_mean'],
        xerr=perm_df['importance_std'], color='steelblue', edgecolor='black')
ax.set_xlabel('Diminution moyenne de l\'accuracy')
ax.set_title('Importance des variables par permutation')
ax.axvline(x=0, color='red', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('figures/importance_variables.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Figure 3 générée : importance_variables.png")

# Figure 4 : PDP
fig, ax = plt.subplots(figsize=(12, 4))
PartialDependenceDisplay.from_estimator(
    best_pipe, X_test, features=['age', 'fare', 'pclass'],
    feature_names=FEATURES, ax=ax, kind='average', grid_resolution=30
)
plt.suptitle('Partial Dependence Plots — Effet des variables sur la survie', fontsize=13)
plt.tight_layout()
plt.savefig('figures/pdp_titanic.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Figure 4 générée : pdp_titanic.png")

# Figure 5 : Matrice de confusion - Textes
fig, ax = plt.subplots(figsize=(7,6))
ConfusionMatrixDisplay.from_predictions(y_text_test, y_pred_best_text,
                                         display_labels=[c.split('.')[-1] for c in news_test.target_names],
                                         ax=ax, cmap='Blues')
ax.set_title(f'Matrice de confusion - {best_text_name}')
plt.tight_layout()
plt.savefig('figures/matrice_confusion_texte.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Figure 5 générée : matrice_confusion_texte.png")

# Figure 6 : Mots discriminants
fig, axes = plt.subplots(2, 2, figsize=(14,10))
for i, class_name in enumerate(news_train.target_names):
    coef = clf.coef_[i]
    top_idx = np.argsort(coef)[-15:]
    axes.flat[i].barh([feature_names_text[j] for j in top_idx], coef[top_idx], color='steelblue')
    axes.flat[i].set_title(f'Classe: {class_name.split(".")[-1]}')
    axes.flat[i].set_xlabel('Coefficient logistique')
plt.suptitle('Top 15 mots discriminants par classe', fontsize=13)
plt.tight_layout()
plt.savefig('figures/mots_discriminants.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Figure 6 générée : mots_discriminants.png")

# Figure 7 : Effet des n-grammes
ngram_results = {}
ngram_ranges = [(1,1), (1,2), (1,3), (2,2)]
print("\n📊 Calcul de l'effet des n-grammes...")

for ngram_range in ngram_ranges:
    pipe_ngram = Pipeline([
        ('vect', TfidfVectorizer(max_features=15000, stop_words='english', ngram_range=ngram_range)),
        ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ])
    scores = cross_val_score(pipe_ngram, X_text_train, y_text_train, cv=3, scoring='accuracy', n_jobs=-1)
    ngram_results[str(ngram_range)] = scores.mean()

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12']
bars = ax.bar(list(ngram_results.keys()), list(ngram_results.values()),
              color=colors, edgecolor='black', linewidth=1.5)

for bar, val in zip(bars, ngram_results.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylabel('Accuracy (CV 3-fold)', fontsize=12)
ax.set_xlabel('Plage de n-grammes', fontsize=12)
ax.set_title('Effet des n-grammes sur la performance', fontsize=13, fontweight='bold')
ax.set_ylim(0.85, 0.95)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/effet_ngrammes.png', dpi=150, bbox_inches='tight')
plt.close()
print("✅ Figure 7 générée : effet_ngrammes.png")

# ------------------------------------------------------------------
# 4.7 Création du ZIP des figures
# ------------------------------------------------------------------
with zipfile.ZipFile('figures_completes.zip', 'w') as zipf:
    for f in os.listdir('figures'):
        zipf.write(os.path.join('figures', f), arcname=f)

print("\n" + "="*50)
print("📁 Figures sauvegardées dans 'figures/':")
for f in os.listdir('figures'):
    print(f"   - {f}")
print("="*50)


# ============================================================
# 5. EXPORT DES SORTIES EXPÉRIMENTALES
# ============================================================
print("\n" + "="*60)
print("EXPORT DES SORTIES EXPÉRIMENTALES")
print("="*60)

outputs = {
    "date": str(datetime.now()),
    "partie_I": {
        "train_shape": list(X_train.shape),
        "test_shape": list(X_test.shape),
        "cv_logreg": float(cross_val_score(pipe_logreg, X_train, y_train, cv=5).mean()),
        "cv_rf": float(cross_val_score(pipe_rf, X_train, y_train, cv=5).mean()),
        "best_params": str(grid_search.best_params_),
        "best_cv_score": float(grid_search.best_score_),
        "test_accuracy_best": float(accuracy_score(y_test, best_pipe.predict(X_test)))
    },
    "partie_II": {
        "train_docs": len(X_text_train),
        "test_docs": len(X_text_test),
        "classes": news_train.target_names,
        "cv_count_nb": float(cross_val_score(pipe_count_nb, X_text_train, y_text_train, cv=3).mean()),
        "cv_tfidf_lr": float(cross_val_score(pipe_tfidf_lr, X_text_train, y_text_train, cv=3).mean()),
        "best_test_accuracy": float(best_acc),
        "best_pipeline_name": best_text_name
    }
}

with open('sorties_experimentales.json', 'w') as f:
    json.dump(outputs, f, indent=2)

print("✅ Fichier 'sorties_experimentales.json' créé")

# Affichage du résumé
print("\n" + "="*50)
print("RÉSUMÉ DES RÉSULTATS")
print("="*50)
print(f"\n📅 Date : {outputs['date']}")
print(f"\n📊 PARTIE I - Titanic :")
print(f"   - LogReg CV : {outputs['partie_I']['cv_logreg']:.4f}")
print(f"   - Random Forest CV : {outputs['partie_I']['cv_rf']:.4f}")
print(f"   - Meilleurs paramètres : {outputs['partie_I']['best_params']}")
print(f"   - Test accuracy : {outputs['partie_I']['test_accuracy_best']:.4f}")
print(f"\n📝 PARTIE II - 20 Newsgroups :")
print(f"   - Count+NB CV : {outputs['partie_II']['cv_count_nb']:.4f}")
print(f"   - TF-IDF+LR CV : {outputs['partie_II']['cv_tfidf_lr']:.4f}")
print(f"   - Meilleur pipeline : {best_text_name}")
print(f"   - Test accuracy : {outputs['partie_II']['best_test_accuracy']:.4f}")
print("\n" + "="*50)
print("✅ PROJET TERMINÉ AVEC SUCCÈS !")
print("="*50)

✅ Tous les imports sont OK

PARTIE I : ANALYSE DU DATASET TITANIC

📊 Dimensions : 891 lignes × 9 colonnes
🎯 Variable cible : 'survived' (0 = décédé, 1 = survivant)

📋 Types de données :
survived      int64
pclass        int64
sex          object
age         float64
sibsp         int64
parch         int64
fare        float64
embarked     object
alone         int64
dtype: object

🔍 Valeurs manquantes :
   - age: 177 (19.87%)
   - embarked: 2 (0.22%)

📁 Split des données :
   - Train : 712 échantillons
   - Test  : 179 échantillons

🏷️ Classification des variables :
   - Numériques : ['age', 'fare', 'sibsp', 'parch']
   - Ordinale   : ['pclass']
   - Nominale   : ['sex', 'embarked']
   - Binaire    : ['alone']

⚙️ Préprocesseur construit avec ColumnTransformer

🚀 3 pipelines définis :
   - Pipeline A : Logistic Regression (baseline)
   - Pipeline B : Random Forest
   - Pipeline C : Random Forest + KNNImputer

📊 Validation croisée 5-fold sur X_train :
   LogReg          : 0.7978 ± 0.0143
 

In [3]:
# ============================================================
# 6. TÉLÉCHARGEMENT DES FICHIERS (Google Colab)
# ============================================================
print("\n" + "="*60)
print("TÉLÉCHARGEMENT DES FICHIERS")
print("="*60)

from google.colab import files

# 1. Télécharger le fichier JSON des sorties expérimentales
if os.path.exists('sorties_experimentales.json'):
    files.download('sorties_experimentales.json')
    print("✅ sorties_experimentales.json téléchargé")
else:
    print("❌ sorties_experimentales.json non trouvé")

# 2. Télécharger le ZIP des figures
if os.path.exists('figures_completes.zip'):
    files.download('figures_completes.zip')
    print("✅ figures_completes.zip téléchargé")
else:
    print("❌ figures_completes.zip non trouvé")

# 3. Télécharger également le ZIP simple des figures
if os.path.exists('figures.zip'):
    files.download('figures.zip')
    print("✅ figures.zip téléchargé")
else:
    print("❌ figures.zip non trouvé")

print("\n" + "="*50)
print("✅ TOUS LES FICHIERS ONT ÉTÉ TÉLÉCHARGÉS !")
print("="*50)


TÉLÉCHARGEMENT DES FICHIERS


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ sorties_experimentales.json téléchargé


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ figures_completes.zip téléchargé
❌ figures.zip non trouvé

✅ TOUS LES FICHIERS ONT ÉTÉ TÉLÉCHARGÉS !
